<a href="https://colab.research.google.com/github/VicenteHeredia9/Laboratorio-Implementaci-n-de-Redes-Neuronales-MLP-MLflow-Vicente/blob/main/actividad_de_laboratorio_de_la_2_12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Actividad:** Laboratorio 2.12

**Nombre:** Vicente heredia

**Asignatura:**: Mineria de datos

**Docente:** Eduardo Farias

In [1]:
!pip install -q mlflow pyngrok

In [2]:
import os, random, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sklearn, mlflow
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, ConfusionMatrixDisplay



In [3]:
SEED = 17
random.seed(SEED)
np.random.seed(SEED)
print(pd.__version__, sklearn.__version__, mlflow.__version__)

2.2.3 1.6.1 3.18.0


In [4]:
import kagglehub
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

ruta = kagglehub.dataset_download("uciml/breast-cancer-wisconsin-data")
print(os.listdir(ruta))

Using Colab cache for faster access to the 'breast-cancer-wisconsin-data' dataset.
['data.csv']


In [5]:
df = pd.read_csv(os.path.join(ruta, "data.csv"))
print(df.shape)
print(df.isna().sum().loc[lambda s: s > 0])

(569, 33)
Unnamed: 32    569
dtype: int64


In [6]:
df = df.drop(columns=["Unnamed: 32", "id"])
df["diagnosis"] = df["diagnosis"].map({"M": 1, "B": 0})
print(df["diagnosis"].value_counts())

diagnosis
0    357
1    212
Name: count, dtype: int64


In [7]:
r = df[["radius_mean", "area_mean", "smoothness_mean"]].agg(["min", "max"]).T
r["rango"] = r["max"] - r["min"]
print(r)

                       min        max       rango
radius_mean        6.98100    28.1100    21.12900
area_mean        143.50000  2501.0000  2357.50000
smoothness_mean    0.05263     0.1634     0.11077


In [8]:
X, y = df.drop(columns=["diagnosis"]), df["diagnosis"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)
print(y_train.mean(), y_test.mean())

0.37362637362637363 0.3684210526315789


In [11]:
import subprocess, time, requests
from pyngrok import ngrok

def servidor_activo():
    try:
        return requests.get("http://127.0.0.1:5000/health", timeout=3).ok
    except requests.RequestException:
        return False

if not servidor_activo():
    env = dict(os.environ, MLFLOW_SERVER_ALLOWED_HOSTS="*", MLFLOW_SERVER_CORS_ALLOWED_ORIGINS="*")
    subprocess.Popen(["mlflow", "server", "--backend-store-uri", "sqlite:///mlflow.db",
                      "--host", "127.0.0.1", "--port", "5000"], env=env)
    for _ in range(60):
        if servidor_activo():
            break
        time.sleep(1)
print("Servidor MLflow activo:", servidor_activo())

try:
    ngrok.set_auth_token(userdata.get("NGROK_AUTHTOKEN"))
except Exception as e:
    print("Authtoken ya configurado, se continúa:", str(e)[:80])

ngrok.kill()
print("MLflow UI:", ngrok.connect(5000).public_url)

Servidor MLflow activo: True
MLflow UI: https://finance-unhidden-duty.ngrok-free.dev


In [12]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("IEI097-L2.12-Heredia")

2026/11/12 00:27:41 INFO mlflow.tracking.fluent: Experiment with name 'IEI097-L2.12-Heredia' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1794443261906, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1794443261906, lifecycle_stage='active', name='IEI097-L2.12-Heredia', tags={}, trace_location=None, workspace='default'>

In [13]:
def correr(nombre, escalar, capas, act, fase, seed=SEED, alpha=1e-4):
    pasos = [("scaler", StandardScaler())] if escalar else []
    pasos.append(("mlp", MLPClassifier(hidden_layer_sizes=capas, activation=act,
                  alpha=alpha, max_iter=500, random_state=seed)))
    pipe = Pipeline(pasos)

    with mlflow.start_run(run_name=nombre):
        mlflow.set_tag("fase", fase)
        mlflow.log_params({"arquitectura": str(capas), "activacion": act, "alpha": alpha,
                           "max_iter": 500, "random_state": seed, "escalado": escalar,
                           "semilla_particion": SEED})
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            pipe.fit(X_train, y_train)
        mlp = pipe.named_steps["mlp"]
        p_tr, p_te = pipe.predict(X_train), pipe.predict(X_test)

        m = {"acc_train": accuracy_score(y_train, p_tr),
             "acc_test": accuracy_score(y_test, p_te),
             "recall_maligno": recall_score(y_test, p_te, pos_label=1),
             "f1_maligno": f1_score(y_test, p_te, pos_label=1),
             "n_iter": mlp.n_iter_}
        mlflow.log_metrics(m)
        for i, l in enumerate(mlp.loss_curve_):
            mlflow.log_metric("train_loss", l, step=i)

        fig, ax = plt.subplots(figsize=(4, 4))
        ConfusionMatrixDisplay.from_predictions(y_test, p_te, ax=ax,
                                                display_labels=["Benigno", "Maligno"])
        mlflow.log_figure(fig, "matriz_confusion.png"); plt.close(fig)
    print(nombre, {k: round(v, 4) for k, v in m.items()})
    return m

In [14]:
mA = correr("A_sin_escalar_relu_30", False, (30,), "relu", "experimentos")

🏃 View run A_sin_escalar_relu_30 at: http://127.0.0.1:5000/#/experiments/1/runs/2bfbc9b64e904e51befc7d94e5525ef6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
A_sin_escalar_relu_30 {'acc_train': 0.9429, 'acc_test': 0.9474, 'recall_maligno': 0.881, 'f1_maligno': 0.925, 'n_iter': 313}


In [15]:
mB = correr("B_escalado_relu_30", True, (30,), "relu", "experimentos")

🏃 View run B_escalado_relu_30 at: http://127.0.0.1:5000/#/experiments/1/runs/9631ee6c22714399ab890afec26847dd
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
B_escalado_relu_30 {'acc_train': 0.9934, 'acc_test': 0.9561, 'recall_maligno': 0.9048, 'f1_maligno': 0.9383, 'n_iter': 352}


In [16]:
mC = correr("C_escalado_relu_64_32", True, (64, 32), "relu", "experimentos")

🏃 View run C_escalado_relu_64_32 at: http://127.0.0.1:5000/#/experiments/1/runs/0c31fcd494694e4cb63c84d030c7154b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
C_escalado_relu_64_32 {'acc_train': 1.0, 'acc_test': 0.9649, 'recall_maligno': 0.9048, 'f1_maligno': 0.95, 'n_iter': 191}


In [17]:
mD = correr("D_escalado_tanh_64_32", True, (64, 32), "tanh", "experimentos")

🏃 View run D_escalado_tanh_64_32 at: http://127.0.0.1:5000/#/experiments/1/runs/4a87939bdc4b41dcab7c33d28c2fe29a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
D_escalado_tanh_64_32 {'acc_train': 1.0, 'acc_test': 0.9649, 'recall_maligno': 0.9048, 'f1_maligno': 0.95, 'n_iter': 284}


In [18]:
res = [correr(f"C_seed{s}", True, (64, 32), "relu", "semillas", seed=s)
       for s in range(SEED, SEED + 5)]

🏃 View run C_seed17 at: http://127.0.0.1:5000/#/experiments/1/runs/3af9553bdcb446238fdcb01508cb9a24
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
C_seed17 {'acc_train': 1.0, 'acc_test': 0.9649, 'recall_maligno': 0.9048, 'f1_maligno': 0.95, 'n_iter': 191}
🏃 View run C_seed18 at: http://127.0.0.1:5000/#/experiments/1/runs/258b0ffbda234a78b1753859af84f4b4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
C_seed18 {'acc_train': 1.0, 'acc_test': 0.9649, 'recall_maligno': 0.9286, 'f1_maligno': 0.9512, 'n_iter': 171}
🏃 View run C_seed19 at: http://127.0.0.1:5000/#/experiments/1/runs/82ef682899a942a49f33fc1017e2c727
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
C_seed19 {'acc_train': 1.0, 'acc_test': 0.9561, 'recall_maligno': 0.9048, 'f1_maligno': 0.9383, 'n_iter': 185}
🏃 View run C_seed20 at: http://127.0.0.1:5000/#/experiments/1/runs/cdf44bd1b8a04fd195250fc0469e3673
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
C_seed20 {'acc_train': 1.0, '

In [19]:
m_rep = correr("C_replica", True, (64, 32), "relu", "replica")
print({k: abs(mC[k] - m_rep[k]) for k in mC})

🏃 View run C_replica at: http://127.0.0.1:5000/#/experiments/1/runs/095d94b76d9348c2a6d51ffd41598f97
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
C_replica {'acc_train': 1.0, 'acc_test': 0.9649, 'recall_maligno': 0.9048, 'f1_maligno': 0.95, 'n_iter': 191}
{'acc_train': 0.0, 'acc_test': 0.0, 'recall_maligno': 0.0, 'f1_maligno': 0.0, 'n_iter': 0}


In [20]:
t = pd.DataFrame(res)[["acc_test", "recall_maligno"]].agg(["min", "max"]).T
t["rango"] = t["max"] - t["min"]
print(t)

                     min       max     rango
acc_test        0.956140  0.973684  0.017544
recall_maligno  0.904762  0.928571  0.023810


In [21]:
runs = mlflow.search_runs(experiment_names=["IEI097-L2.12-Heredia"])
cols = ["tags.mlflow.runName", "tags.fase", "params.arquitectura", "params.activacion",
        "params.escalado", "params.random_state", "params.semilla_particion",
        "metrics.acc_train", "metrics.acc_test", "metrics.recall_maligno",
        "metrics.f1_maligno", "metrics.n_iter"]
runs[cols].sort_values("metrics.recall_maligno", ascending=False).to_csv("runs.csv", index=False)
print(len(runs))

10


In [22]:
from google.colab import files
files.download("runs.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
ngrok.kill()